### Imports

In [1]:
import chipwhisperer as cw
import matplotlib.pyplot as plt
import numpy as np
import time
import struct
import random

from scipy.signal import find_peaks

In [2]:
# project_name = "protected1mil"
# num_traces = 1000000
# scmd_value = 1 #0 for unprotected and 1 for protected

In [32]:
project_name = "protected_n1_2_traces_5k"
num_traces = 5000
scmd_value = 0 #0 for unprotected and 1 for protected

In [3]:
min_in_val = -2
max_in_val = 2
decimate_value = 1
# decimate value read about for chipwhisperer

filename = project_name + "-trace.txt"

### Function Definitions

In [4]:
def random_float(min_val, max_val):
    # Generate a random float between min_val and max_val
    rand_float = random.uniform(min_val, max_val)
    # Round to 2 decimal places
    return round(rand_float, 2)

In [5]:
def float_to_bytearray_32bit_little_edian(f):
    # Pack the float as a 32-bit (4-byte) IEEE 754 floating point number
    packed = struct.pack('f', f)
    # Convert to bytearray
    return bytearray(packed)

In [6]:
def scope_setup(samples=24431, decimate=2):
    # arm the scope
    scope.arm()
    
    # Set the maximum number of points in a trace
    scope.adc.fifo_fill_mode = "normal"
    scope.adc.samples = samples
    scope.adc.decimate = decimate

In [7]:
def capture_trace(cmd_data, cmd='p', scmd=scmd_value, prints=True):
    scope.arm()
    # flush the UART buffer
    target.flush()
    
    target.send_cmd(cmd, scmd, cmd_data)
    ret = scope.capture()
    trace = scope.get_last_trace()
    
    returned_data = target.read_cmd('r')
    ack = target.read_cmd('e')
    if prints:
        print(f'r\t- target.read_cmd("r"):\t{returned_data}')
        print(f'ack\t- target.read_cmd("e"):\t{ack}')
    return trace
    

### Target Setup

In [8]:
#Scope setup
scope = cw.scope() 
scope.default_setup()

target = cw.target(scope, cw.targets.SimpleSerial2) #cw.targets.SimpleSerial can be omitted
#MY CHANGES - changed target to SimpleSerial2 - to be able to send_cmd

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 836441                    to 22125746                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0               
scope.clock.clkgen_

In [9]:
def disconnect_target():
    """Properly disconnect the target"""
    try:
        # Send disconnect command
        target.send_cmd('d', 0, b'')
        
        # Read final acknowledgement
        returned_data = target.read_cmd('r', timeout=100)
        print(f"Disconnect ACK: {returned_data}")
        
        # Short delay to ensure transmission
        time.sleep(0.1)
        
    except Exception as e:
        print(f"Disconnect warning: {e}")
    
    finally:
        # Always disconnect scope and target
        target.dis()
        scope.dis()
        print("Target and scope disconnected")

In [38]:
disconnect_target()

(ChipWhisperer Target WARNING|File SimpleSerial2.py:518) Unexpected start to command 100
(ChipWhisperer Target WARNING|File SimpleSerial2.py:540) Unexpected frame byte in CWbytearray(b'00 64 02 65 00 03 65')
(ChipWhisperer Target ERROR|File SimpleSerial2.py:186) Infinite loop in unstuff data
(ChipWhisperer Target ERROR|File SimpleSerial2.py:187) bytearray(b'\x00d\x02e\x00\x03e')
(ChipWhisperer Target WARNING|File SimpleSerial2.py:553) Invalid CRC. Expected 12 got 3
(ChipWhisperer Target WARNING|File SimpleSerial2.py:556) Did not receive end of frame, got 101


Disconnect ACK: CWbytearray(b'00 64 02 65 00 03 65')
Target and scope disconnected


In [10]:
scope_setup(samples=24430, decimate=1)

In [31]:
# Unprotected
# cw.program_target(scope, cw.programmers.STM32FProgrammer, "./compiled_for_CWLITEARM_unprotected_symmetrical/simpleserial-target-CWLITEARM.hex")

# Tables
# cw.program_target(scope, cw.programmers.STM32FProgrammer, "./protected_nn_with_lut_MULT_ADD_TABLES_On/compiled_for_CWLITEARM/simpleserial-target-CWLITEARM.hex")

# Indexed
cw.program_target(scope, cw.programmers.STM32FProgrammer, "./protected_nn_with_lut_MULT_ADD_ARRAY_O1/compiled_for_CWLITEARM/simpleserial-target-CWLITEARM.hex")

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 25187 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 25187 bytes


### Trace collection

### For protected implementations

In [ ]:
trace_waves_arr = []
inputs_arr = []

start = time.time()
completed_counter = 0

ALLOWED_INPUTS = [
    0b110100010000001,
    0b010100010000010,
    0b100000000000011,
    0b100100010000100,
    0b010000000000101,
    0b110000000000110,
    0b000100010000111,
    0b000100010001000,
    0b110000000001001,
    0b010000000001010,
    0b100100010001011,
    0b100000000001100,
    0b010100010001101,
    0b110100010001110
]

warm_quant = [random.choice(ALLOWED_INPUTS) for _ in range(10)]    
warm_bytes = bytearray()
for x in warm_quant:
    warm_bytes += x.to_bytes(2, byteorder='little', signed=False)

for i in range(50):
    trace_wave = capture_trace(warm_bytes, scmd=scmd_value)

print("warm up done")

# QMIN_IN = -7
# QMAX_IN = 7

# Real executions
start = time.time()
completed_counter = 0

trace_waves_arr = []    
inputs_arr = []         

for i in range(num_traces):
    # initialize 10 inputs with 0
    quant_inputs = [0] * 10
    
    # the others remain 0
    quant_inputs[1] = random.choice(ALLOWED_INPUTS)
    
    inputs_arr.append(quant_inputs)
    
    cmd_data = bytearray()
    for x in quant_inputs:
        cmd_data += x.to_bytes(2, byteorder='little', signed=False)

    trace_wave = capture_trace(cmd_data=cmd_data, scmd=scmd_value, prints=False)
    trace_waves_arr.append(np.array(trace_wave))
    
    completed_counter += 1
    if completed_counter % 100 == 0:
        print(f"captured {completed_counter} traces in {time.time() - start:.2f} seconds")

end = time.time()
print(f'Capturing finished in {end - start:.2f} seconds!')

trace_waves_arr = np.array(trace_waves_arr)
inputs_arr = np.array(inputs_arr)

r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 4e 06 00 00 4f 00')
ack	- target.r

### For unprotected implementation

In [ ]:
import random
import numpy as np
import time
import os

def build_cmd(input_vals):
    """Convert list of signed ints (range -128..127) to bytearray for transmission."""
    return bytearray([x & 0xFF for x in input_vals])

def verify_input_integrity(input_vals):
    """Send input with debug command 'd' and check if returned bytes match."""
    cmd = build_cmd(input_vals)
    target.flush()
    target.send_cmd('d', scmd_value, cmd)
    ret = target.read_cmd('d')
    _ = target.read_cmd('e')
    if len(ret) >= 13:   # 0x00, 'd', len (10), 10 data bytes,
        payload = ret[3:13]   # extract the 10 data bytes
        # Convert unsigned bytes back to signed int8
        received = [b if b < 128 else b - 256 for b in payload]
        return input_vals == received
    return False

def get_network_output(input_vals):
    """Send normal command 'p' and return the int32 result."""
    cmd = build_cmd(input_vals)
    target.flush()
    target.send_cmd('p', scmd_value, cmd)
    ret = target.read_cmd('r')
    _ = target.read_cmd('e')
    if len(ret) >= 7:   # 0x00, 'r', len (4), 4 data bytes, CRC?
        return int.from_bytes(ret[3:7], 'little', signed=True)
    return None

def save_files(folder, traces, input_file, input_vectors):
    """Save traces (list of arrays) and input vectors to text files."""
    isExist = os.path.exists(folder)
    if not isExist:
        os.makedirs(folder)
    num_traces = len(traces)
    for n in range(num_traces):
        with open(folder + "/trace_"+str(n)+".txt", "w") as f:
            for sample in traces[n]:
                f.write(str(sample) + "\n")
    with open(folder + "/" + input_file, "w") as f:
        for vec in input_vectors:
            line = " ".join(str(x) for x in vec)
            f.write(line + "\n")
    print(f"Saved {num_traces} trace(s) and inputs to folder '{folder}/'")

NUM_TRACES = 5000          # number of traces for CPA attack
ALLOWED_INPUTS = [-7, -6, -5, -4, -3, -2, -1, 1, 2, 3, 4, 5, 6, 7]

print("Starting CPA trace capture...")
start_time = time.time()

# 1. Warm‑up (50 dummy traces with random inputs)
warm_quant = [random.choice(ALLOWED_INPUTS) for _ in range(10)]
warm_bytes = build_cmd(warm_quant)
for i in range(50):
    capture_trace(warm_bytes, scmd=scmd_value)
print("Warm‑up done (50 traces).")

# 2. Verify data integrity (requires firmware debug command 'd')
test_inputs = [7] + [0]*9
if verify_input_integrity(test_inputs):
    print("Data integrity OK")
else:
    print("Data integrity FAILED – check firmware debug command 'd'")
    exit(1)

# 3. (Optional) Show actual network output for the fixed pattern
actual_out = get_network_output(test_inputs)
print(f"Network output for [7,0,...,0] = {actual_out}")

# 4. Capture N traces with random input[0] and others=0
print(f"\nCapturing {NUM_TRACES} traces for CPA attack...")
trace_waves_arr = []
inputs_arr = []          # store full 10-element input vectors

for i in range(NUM_TRACES):
    # Choose random input[0] from allowed values (non-zero)
    input0 = random.choice(ALLOWED_INPUTS)
    # Build the full 10-element input vector (inputs 1..9 = 0)
    quant_inputs = [0]*1 + [input0] + [0]*8
    inputs_arr.append(quant_inputs)
    
    # Convert to command bytes and capture trace
    cmd_data = build_cmd(quant_inputs)
    trace = capture_trace(cmd_data, scmd=scmd_value, prints=False)
    trace_waves_arr.append(np.array(trace))
    
    if (i+1) % 100 == 0:
        elapsed = time.time() - start_time
        print(f"Captured {i+1}/{NUM_TRACES} traces in {elapsed:.2f} seconds")

end_time = time.time()
print(f"\nCapture finished in {end_time - start_time:.2f} seconds")

save_files("cpa_traces_unprot_n1_2_5k", trace_waves_arr, "inputs.txt", inputs_arr)
print(f"Saved {NUM_TRACES} traces and input vectors in folder 'cpa_traces/'")

Starting CPA trace capture...
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 11 00 00 08 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 11 00 00 08 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 11 00 00 08 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 11 00 00 08 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 11 00 00 08 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 11 00 00 08 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 11 00 00 08 00')
ack	- target.read_cmd("e"):	CWbytearray(b'00 65 01 00 eb 00')
r	- target.read_cmd("r"):	CWbytearray(b'00 72 04 a6 1

In [37]:
save_files(project_name, trace_waves_arr, "inputs.txt", inputs_arr)

Saved 5000 trace(s) and inputs to folder 'protected_n1_2_traces_5k/'


In [19]:
import os

def save_files(folder, array, input_file, input_array):
    isExist = os.path.exists(folder)
    if not isExist:
        os.makedirs(folder)
    no_of_traces = len(input_array)
    for n in range(no_of_traces):
        with open(folder + "/trace_"+str(n)+".txt","w+") as file:
            for record in array[n]:
                file.write(str(record)+"\n")
        file.close()
    
    with open(folder + "/" + input_file,"w+") as file:
        for i in range(no_of_traces):
            line = " ".join([str(x) for x in input_array[i]])
            file.write(line + "\n")
    return